In [1]:
import numpy as np
import pandas as pd
import xarray as xr

%matplotlib inline
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from matplotlib import colors
from matplotlib.patches import Rectangle
from matplotlib.tri import Triangulation

In [2]:
grid_data = {}

# Grid info for deterministic runs:
grid_data["grid_det"] = xr.open_dataset("./grids/icon_grid_0026_R03B07_G.nc")
#grid_data["tri_det"] = Triangulation(np.rad2deg(grid_data["grid_det"]["clon"]), np.rad2deg(grid_data["grid_det"]["clat"]))
grid_data["area_det"] = grid_data["grid_det"]["cell_area"]

# Grid info for ensemble runs:
grid_data["grid_ens"] = xr.open_dataset("./grids/icon_grid_0028_R02B07_N02.nc")
#grid_data["tri_ens"] = Triangulation(np.rad2deg(grid_data["grid_ens"]["clon"]), np.rad2deg(grid_data["grid_ens"]["clat"]))
grid_data["area_ens"] = grid_data["grid_ens"]["cell_area"]


# Define "study area" by cropping deterministic global to ensemble nest
lon_min = np.min(np.rad2deg(grid_data["grid_ens"]["clon"]))
lon_max = np.max(np.rad2deg(grid_data["grid_ens"]["clon"]))

lat_min = np.min(np.rad2deg(grid_data["grid_ens"]["clat"]))
lat_max = np.max(np.rad2deg(grid_data["grid_ens"]["clat"]))

# Crop domain by setting the cell areas to 0
grid_data["cells_det"] = ((np.rad2deg(grid_data["grid_det"]["clon"]) >= lon_min) & (np.rad2deg(grid_data["grid_det"]["clon"]) <= lon_max) & 
                          (np.rad2deg(grid_data["grid_det"]["clat"]) >= lat_min) & (np.rad2deg(grid_data["grid_det"]["clat"]) <= lat_max)).values
grid_data["area_det"][dict(cell=grid_data["grid_det"]["cell"][~grid_data[f"cells_det"]])] = 0

In [3]:
def load_ensemble(datetime, run_names, exp_name, variable):
    # Deterministic:
    ds_det = xr.open_dataset(f"./data/{exp_name}/{datetime}/{variable}_det.nc")[variable].squeeze()
    #ds_det[dict(ncells=grid_data["grid_det"]["cell"][~grid_data[f"cells_det"]])] = np.nan
    #ds_det = ds_det.isel(ncells=grid_data[f"cells_det"])

    ds_det_weighted = ds_det.weighted(grid_data["area_det"].rename({"cell": "ncells"}))#.isel(ncells=grid_data[f"cells_det"]))
    ts_det = ds_det_weighted.mean(dim="ncells", skipna=True)
    
    if variable == "TOT_PREC": #de-accumulate
        ts_det = ts_det.diff(dim="time")
   
    dataframe = pd.Series(ts_det.data, ts_det["time"], name=f"{exp_name}_det").to_frame()
    
    # Ensemble:
    for mem in run_names[:]: #[:] creates a copy, which is looped over
        # Catch missing data due to crashed ensemble member:
        try:
            ds_mem = xr.open_dataset(f"./data/{exp_name}/{datetime}/{variable}_{mem}.nc")[variable].squeeze()
        except:
            run_names.remove(mem)
            continue
        
        ds_mem_weighted = ds_mem.weighted(grid_data["area_ens"].rename({"cell": "ncells"}))
        ts_mem = ds_mem_weighted.mean(dim="ncells", skipna=True)
        
        if variable == "TOT_PREC": #de-accumulate
            ts_mem = ts_mem.diff(dim="time")

        dataframe[f"{exp_name}_{mem}"] = pd.Series(ts_mem.data, ts_mem["time"])

    return dataframe

In [4]:
ini_dates = [2021070912, 2021071012, 2021071112]#, 2021071212]
datetime = ini_dates[0]

In [7]:
exp_names = ["CTRL", "SATU", "WILT", "CO2x2"]
variables = ["T_2M", "TOT_PREC"]
run_names = [f"mem{i:03d}" for i in range(1,21)]

for variable in variables:
    dframes = []
    
    for exp_name in exp_names:
        dframes.append(load_ensemble(datetime, run_names, exp_name, variable))
    
    with open(f"july21_{variable.lower()}.csv", "w") as file:
        if variable == "T_2M":
            file.write("# Area weighted domain average of two meter temperature given in K\n")
        elif variable == "TOT_PREC":
            file.write("# Area weighted domain average of de-accumulated total precipitation given in kg per m2\n")

        pd.concat(dframes, axis=1, join="outer").to_csv(file)

If you save some of the concatenated dataframes as variable dframe, this would be the way to plot:

In [42]:
dframe = pd.concat(dframes, axis=1, join="outer")

In [ ]:
for exp_name, color in zip(exp_names, ["tab:blue", "tab:orange", "tab:green", "tab:red"]):
    plt.plot(dframe[f"{exp_name}_det"], color=color, label=exp_name)
    for name in [name for name in dframe.columns if f"{exp_name}_mem" in name]:
        plt.plot(dframe[name], color=color, alpha=0.5)
plt.legend()

And the PRUDENCE regions:



    ‘BI’: Latitude: <50.0 or >59.0, Longitude: <-10.0 or >2.0

    ‘IP’: Latitude: <36.0 or >44.0, Longitude: <-10.0 or >3.0

    ‘FR’: Latitude: <44.0 or >50.0, Longitude: <-5.0 or >5.0

    ‘ME’: Latitude: <48.0 or >55.0, Longitude: <2.0 or >16.0

    ‘SC’: Latitude: <55.0 or >70.0, Longitude: <5.0 or >30.0

    ‘AL’: Latitude: <44.0 or >48.0, Longitude: <5.0 or >15.0

    ‘MD’: Latitude: <36.0 or >44.0, Longitude: <3.0 or >25.0

    ‘EA’: Latitude: <44.0 or >55.0, Longitude: <16.0 or >30.0


In [16]:
ds_det = xr.open_dataset(f"./data/{exp_name}/{datetime}/TOT_PREC_det.nc")["TOT_PREC"]

In [85]:
ds_det_weighted = ds_det.weighted(grid_data["area_det"].rename({"cell": "ncells"}))
ts_det = ds_det_weighted.mean(dim="ncells", skipna=True).diff(dim="time")
series_det = pd.Series(ts_det.data, ts_det["time"], name=f"{exp_name}_det")

In [91]:
dataframe = series_det.to_frame()

In [19]:
mem = run_names[0]

In [18]:
run_names = [f"mem{i:03d}" for i in range(1,21)]

In [20]:
ds_mem = xr.open_dataset(f"./data/{exp_name}/{datetime}/{variable}_{mem}.nc")[variable]